# 03 - Metaheurísticas (GA + PSO) - Telemetry Heart AI

Usamos dos metaheurísticas para optimizar el proceso de selección de features y tuning de hiperparámetros:
- **`GeneticEngine`** (DEAP): selección de features.
- **`PSOEngine`**: tuning de hiperparámetros del RandomForest.

Ambos operan sobre el dataset **UCI Heart Disease** real (`app/data/heart.csv`,
13 features + `target` binario), que es para el que fueron diseñados
(`n_features=13`). Descárgalo con `python scripts/download_heart.py` si falta.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('cwd:', Path.cwd())

cwd: C:\Users\HALO\Documents\universidad\Inteligentes\telemetry-heart-ai\Project\microservice


In [2]:
from app.services.genetic_engine import GeneticEngine
from app.services.pso_engine import PSOEngine

df = pd.read_csv('app/data/heart.csv')
X = df.drop('target', axis=1)
y = df['target']
print('Dataset:', X.shape, '| target:', y.value_counts().to_dict())

C:\Users\HALO\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: (303, 13) | target: {0: 164, 1: 139}


In [3]:
# --- Algoritmo Genético: selección de features ---
print('Ejecutando Algoritmo Genético...')
ga = GeneticEngine(n_features=X.shape[1], population_size=50, generations=20)
ga_result = ga.run(X, y)
print('Features seleccionadas:', ga_result['feature_names'])
print('N seleccionadas:', ga_result['n_selected'], 'de', X.shape[1])
print('Mejor fitness (F1):', round(ga_result['best_fitness'], 4))

Ejecutando Algoritmo Genético...


Features seleccionadas: ['age', 'sex', 'cp', 'trestbps', 'restecg', 'thalach', 'exang', 'ca', 'thal']
N seleccionadas: 9 de 13
Mejor fitness (F1): 0.9181


In [4]:
# --- PSO: tuning de hiperparámetros del RandomForest ---
print('Ejecutando PSO...')
pso = PSOEngine(n_particles=30, n_iterations=30)
pso_result = pso.run(X, y)
print('Mejores hiperparámetros:', pso_result['best_params'])
print('Mejor fitness (F1):', round(pso_result['best_fitness'], 4))

Ejecutando PSO...


Mejores hiperparámetros: {'n_estimators': 12, 'max_depth': 2, 'min_samples_split': 7, 'min_samples_leaf': 4}
Mejor fitness (F1): 0.9345


In [5]:
# --- Curvas de convergencia ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GA: logbook de DEAP
gens = ga.history.select('gen')
ga_max = ga.history.select('max')
axes[0].plot(gens, ga_max, 'b-o', ms=4)
axes[0].set_title('Convergencia GA (selección de features)')
axes[0].set_xlabel('Generación'); axes[0].set_ylabel('Mejor F1')
axes[0].grid(alpha=0.3)

# PSO: history propio
iters = [h['iteration'] for h in pso.history]
pso_best = [h['best_fitness'] for h in pso.history]
axes[1].plot(iters, pso_best, 'r-s', ms=4)
axes[1].set_title('Convergencia PSO (hiperparámetros)')
axes[1].set_xlabel('Iteración'); axes[1].set_ylabel('Mejor F1')
axes[1].grid(alpha=0.3)

Path('app/data/charts').mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig('app/data/charts/convergence.png', dpi=120)
plt.show()

C:\Users\HALO\AppData\Local\Temp\ipykernel_26800\1650945643.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# --- Features seleccionadas por el GA ---
selected = [1 if i in ga_result['selected_features'] else 0 for i in range(X.shape[1])]
plt.figure(figsize=(10, 5))
colors = ['#1f77b4' if s else '#cccccc' for s in selected]
plt.bar(X.columns, selected, color=colors)
plt.title('Features seleccionadas por el GA (1 = seleccionada)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('app/data/charts/ga_features.png', dpi=120)
plt.show()
print('Metaheurísticas (GA + PSO) completas')

Metaheurísticas (GA + PSO) completas


C:\Users\HALO\AppData\Local\Temp\ipykernel_26800\1450758424.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
